In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import warnings
warnings.filterwarnings('ignore')

CREDITS_PATH = "credits.csv"
MOVIES_PATH = "movies_metadata.csv"

try:
    credits_df = pd.read_csv(CREDITS_PATH)
    print("Файл credits.csv успешно загружен")
except FileNotFoundError:
    print(f"Ошибка: файл '{CREDITS_PATH}' не найден")
    print("Убедитесь, что файл находится в той же директории")
    exit(1)
except Exception as e:
    print(f"Ошибка при загрузке: {e}")
    exit(1)

try:
    movies_df = pd.read_csv(MOVIES_PATH, low_memory=False)
    print("Файл movies_metadata.csv успешно загружен")
except FileNotFoundError:
    print(f"Ошибка: файл '{MOVIES_PATH}' не найден")
    print("Скачайте оба файла из датасета The Movies Dataset")
    exit(1)
except Exception as e:
    print(f"Ошибка при загрузке: {e}")
    exit(1)

print(f"\nCredits shape: {credits_df.shape}")
print(f"Movies shape: {movies_df.shape}")

try:
    movies_df['id'] = movies_df['id'].astype(str)
    credits_df['id'] = credits_df['id'].astype(str)

    merged_df = movies_df[['id', 'title', 'vote_average']].merge(
        credits_df[['id', 'crew', 'genres']],
        on='id',
        how='inner'
    )
    print(f"Объединенный датасет: {merged_df.shape}")
except KeyError as e:
    print(f"Ошибка: отсутствует необходимая колонка - {e}")
    print("Проверьте структуру загруженных файлов")
    exit(1)

def extract_director(crew_str):
    try:
        if pd.isna(crew_str) or crew_str == '':
            return None
        crew_list = ast.literal_eval(crew_str)
        for member in crew_list:
            if member.get('job') == 'Director':
                return member.get('name')
        return None
    except (ValueError, SyntaxError, TypeError):
        return None

def extract_genres(genres_str):
    try:
        if pd.isna(genres_str) or genres_str == '':
            return []
        genres_list = ast.literal_eval(genres_str)
        return [g.get('name') for g in genres_list if g.get('name')]
    except (ValueError, SyntaxError, TypeError):
        return []

print("\nИзвлечение информации о режиссерах и жанрах...")
merged_df['director'] = merged_df['crew'].apply(extract_director)
merged_df['genres_list'] = merged_df['genres'].apply(extract_genres)

merged_df = merged_df[merged_df['director'].notna()]
merged_df = merged_df[merged_df['genres_list'].apply(lambda x: len(x) > 0)]
merged_df = merged_df[merged_df['vote_average'].notna()]

print(f"После фильтрации: {merged_df.shape}")

result_df = merged_df[['title', 'director', 'genres_list', 'vote_average']].copy()
print(f"\nРезультирующий датасет: {result_df.shape[0]} фильмов")

print("\nПример данных:")
result_df.head(10)

director_stats = result_df.groupby('director')['vote_average'].agg(['mean', 'median', 'count'])
director_stats = director_stats[director_stats['count'] >= 3]
director_stats = director_stats.sort_values('mean', ascending=False)

print("\n" + "="*60)
print("ТОП-10 РЕЖИССЕРОВ С ВЫСОКИМ РЕЙТИНГОМ")
print("="*60)
print(director_stats.head(10))

print("\n" + "="*60)
print("ТОП-10 РЕЖИССЕРОВ С НИЗКИМ РЕЙТИНГОМ")
print("="*60)
print(director_stats.tail(10))

exploded_df = result_df.explode('genres_list')
exploded_df = exploded_df.rename(columns={'genres_list': 'genre'})

genre_stats = exploded_df.groupby('genre')['vote_average'].agg(['mean', 'median', 'count'])
genre_stats = genre_stats[genre_stats['count'] >= 5]
genre_stats = genre_stats.sort_values('mean', ascending=False)

print("\n" + "="*60)
print("ТОП-10 ЖАНРОВ С ВЫСОКИМ РЕЙТИНГОМ")
print("="*60)
print(genre_stats.head(10))

print("\n" + "="*60)
print("ТОП-10 ЖАНРОВ С НИЗКИМ РЕЙТИНГОМ")
print("="*60)
print(genre_stats.tail(10))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].barh(director_stats.head(10).index[::-1], director_stats.head(10)['mean'][::-1])
axes[0, 0].set_title('Топ-10 режиссеров по среднему рейтингу')
axes[0, 0].set_xlabel('Средний рейтинг')

axes[0, 1].barh(director_stats.tail(10).index[::-1], director_stats.tail(10)['mean'][::-1])
axes[0, 1].set_title('Топ-10 режиссеров с низким рейтингом')
axes[0, 1].set_xlabel('Средний рейтинг')
axes[0, 1].set_xlim(axes[0, 0].get_xlim())

axes[1, 0].barh(genre_stats.head(10).index[::-1], genre_stats.head(10)['mean'][::-1])
axes[1, 0].set_title('Топ-10 жанров по среднему рейтингу')
axes[1, 0].set_xlabel('Средний рейтинг')

axes[1, 1].barh(genre_stats.tail(10).index[::-1], genre_stats.tail(10)['mean'][::-1])
axes[1, 1].set_title('Топ-10 жанров с низким рейтингом')
axes[1, 1].set_xlabel('Средний рейтинг')

plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.histplot(director_stats['mean'], bins=20, kde=True)
plt.title('Распределение средних рейтингов режиссеров')
plt.xlabel('Средний рейтинг')

plt.subplot(1, 2, 2)
sns.histplot(genre_stats['mean'], bins=20, kde=True, color='orange')
plt.title('Распределение средних рейтингов жанров')
plt.xlabel('Средний рейтинг')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("ИТОГОВЫЙ ДАТАСЕТ ДЛЯ ЭКСПОРТА")
print("="*60)
result_export = result_df[['title', 'director', 'vote_average']].copy()
result_export['genres'] = result_df['genres_list'].apply(lambda x: ', '.join(x))
print(f"Размер: {result_export.shape}")
print("\nПример итоговых данных:")
result_export.head(10)

result_export.to_csv('movies_directors_genres.csv', index=False)
print("\nДатасет сохранен как 'movies_directors_genres.csv'")

print("\n" + "="*60)
print("ВЫВОДЫ")
print("="*60)
print(f"1. Проанализировано {len(director_stats)} режиссеров (минимум 3 фильма)")
print(f"2. Проанализировано {len(genre_stats)} жанров")
print(f"3. Самый высокий средний рейтинг у режиссера: {director_stats.index[0]} ({director_stats.iloc[0]['mean']:.2f})")
print(f"4. Самый низкий средний рейтинг у режиссера: {director_stats.index[-1]} ({director_stats.iloc[-1]['mean']:.2f})")
print(f"5. Самый высокий средний рейтинг у жанра: {genre_stats.index[0]} ({genre_stats.iloc[0]['mean']:.2f})")
print(f"6. Самый низкий средний рейтинг у жанра: {genre_stats.index[-1]} ({genre_stats.iloc[-1]['mean']:.2f})")

Ошибка: файл 'credits.csv' не найден
Убедитесь, что файл находится в той же директории
Ошибка: файл 'movies_metadata.csv' не найден
Скачайте оба файла из датасета The Movies Dataset


NameError: name 'credits_df' is not defined